# Transfer Learning - NIH Chest X-Ray (PyDrive2 Setup)

**✨ No drive.mount() needed!**

Uses PyDrive2 for fast, cached authentication to Google Drive.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Install and Authenticate PyDrive2

**This uses Colab's built-in auth - no OAuth prompts!**

In [ ]:
!pip install -q pydrive2

from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

# Authenticate (uses Colab's built-in auth)
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

print("✓ PyDrive2 authenticated")
print("✓ No drive.mount() needed!")

## 3. Download Data from Google Drive

**⚠️ Update DATA_FOLDER_ID with your Drive folder ID**

Get it from Drive URL: `https://drive.google.com/drive/folders/YOUR_FOLDER_ID`

In [ ]:
from pathlib import Path

def download_folder_from_drive(drive, folder_id, destination_dir):
    """Download all files from a Google Drive folder."""
    destination_dir = Path(destination_dir)
    destination_dir.mkdir(parents=True, exist_ok=True)
    
    # List files in folder
    file_list = drive.ListFile({
        'q': f"'{folder_id}' in parents and trashed=false"
    }).GetList()
    
    print(f"Found {len(file_list)} files in Drive folder")
    
    # Download each file
    for file in file_list:
        file_path = destination_dir / file['title']
        print(f"📥 Downloading {file['title']}... ", end='')
        file.GetContentFile(str(file_path))
        size_mb = file_path.stat().st_size / 1024 / 1024
        print(f"✓ ({size_mb:.1f} MB)")
    
    return len(file_list)

# ⚠️ REPLACE WITH YOUR GOOGLE DRIVE FOLDER ID
DATA_FOLDER_ID = 'YOUR_FOLDER_ID_HERE'  # Get from Drive URL

# Download data
destination = Path('/content/data')
num_files = download_folder_from_drive(drive, DATA_FOLDER_ID, destination)

print(f"\n✓ Downloaded {num_files} files to {destination}")

# Verify files
csv_files = list(destination.glob('*.csv'))
json_files = list(destination.glob('*.json'))
print(f"\n📊 CSV files: {[f.name for f in csv_files]}")
print(f"📋 JSON files: {[f.name for f in json_files]}")

## 4. Setup and Configuration

In [ ]:
import json
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from tensorflow.keras.applications import ResNet50, DenseNet121, EfficientNetB3
from tensorflow.keras.preprocessing.image import ImageDataGenerator

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPUs available: {len(tf.config.list_physical_devices('GPU'))}")

In [ ]:
# Setup directories
PROJECT_ROOT = Path('/content')
PROCESSED_DIR = Path('/content/data')
MODELS_DIR = PROJECT_ROOT / 'models'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Data directory: {PROCESSED_DIR}")
print(f"✓ Models directory: {MODELS_DIR}")

In [ ]:
# Configuration
CONFIG = {
    'img_height': 224,
    'img_width': 224,
    'channels': 3,
    'batch_size': 64,
    'epochs_stage1': 5,
    'epochs_stage2': 10,
    'learning_rate_stage1': 0.001,
    'learning_rate_stage2': 0.0001,
    'unfreeze_layers': 20,
    'dense_units': 512,
    'dropout_rate': 0.5,
    'early_stopping_patience': 10,
    'reduce_lr_patience': 5,
    'num_classes': 14,
    'use_sample': True,
    'sample_size': 1000,
    'random_state': 42
}

MODELS_TO_TRAIN = ['resnet50', 'densenet121', 'efficientnetb3']

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## 5. Load Data

In [ ]:
# Load CSV files
train_df = pd.read_csv(PROCESSED_DIR / 'train_split.csv')
val_df = pd.read_csv(PROCESSED_DIR / 'val_split.csv')
test_df = pd.read_csv(PROCESSED_DIR / 'test_split.csv')

# Load preprocessing config
with open(PROCESSED_DIR / 'preprocessing_config.json', 'r') as f:
    prep_config = json.load(f)

disease_classes = prep_config['disease_classes']

print(f"✓ Train: {len(train_df):,} images")
print(f"✓ Val:   {len(val_df):,} images")
print(f"✓ Test:  {len(test_df):,} images")
print(f"✓ Disease classes ({len(disease_classes)}): {disease_classes}")

## 6. Upload Results Back to Drive (After Training)

**Run this cell after training to save results to Drive**

In [ ]:
def upload_file_to_drive(drive, local_path, drive_folder_id):
    """Upload a file to Google Drive."""
    local_path = Path(local_path)
    
    file_metadata = {
        'title': local_path.name,
        'parents': [{'id': drive_folder_id}]
    }
    
    file = drive.CreateFile(file_metadata)
    file.SetContentFile(str(local_path))
    file.Upload()
    
    print(f"✓ Uploaded: {local_path.name}")
    print(f"  URL: https://drive.google.com/file/d/{file['id']}")
    return file['id']

# Upload trained models
RESULTS_FOLDER_ID = DATA_FOLDER_ID  # Or create a separate results folder

for model_file in MODELS_DIR.glob('*.keras'):
    upload_file_to_drive(drive, model_file, RESULTS_FOLDER_ID)

print("\n✓ All results uploaded to Drive")

---

## Continue with your training code...

Add your model building, training, and evaluation cells below.